# Wildfire Detection & Monitoring
**PyGeoVision v2.0 — Notebook 16**

Detect wildfires and classify burn severity using BAI + dNBR (USFS protocol).

```bash
pip install "pygeovision[geo,train]"
```

In [ ]:
import pygeovision as pgv
import numpy as np, rasterio, matplotlib.pyplot as plt
import matplotlib.colors as mcolors, matplotlib.patches as mpatches
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE_DIR = Path("./outputs/16_wildfire"); BASE_DIR.mkdir(parents=True,exist_ok=True)
BBOX = (-120.5, 39.0, -119.5, 40.0)
client = pgv.PyGeoVision()
print(client); print("\nStudy: Sierra Nevada, California")
print("Indices: BAI + dNBR | USFS 4-class severity classification")

In [ ]:
def get_s(dr,label):
    r=client.search(bbox=BBOX,date_range=dr,providers=["planetary_computer"],cloud_cover_max=20,limit=3)
    print(f"  {label}: {len(r)} scenes")
    if r:
        dl=client.download(r[:1],output_dir=BASE_DIR/label.replace(" ","_"),
                            post_process=["reproject:EPSG:32610","cog"])
        if dl and dl[0].success: return dl[0].path
    return None

pre  = get_s(("2024-07-01","2024-08-01"),"pre fire")
post = get_s(("2024-09-01","2024-10-01"),"post fire")

In [ ]:
np.random.seed(55); H=W=512
pre_nir=np.random.uniform(0.40,0.80,(H,W)); pre_red=np.random.uniform(0.05,0.15,(H,W))
pre_swir=np.random.uniform(0.10,0.25,(H,W))
post_nir=pre_nir.copy(); post_red=pre_red.copy(); post_swir=pre_swir.copy()
fire_mask=np.zeros((H,W),dtype=bool)
rng=np.random.default_rng(55)
for cy,cx,ry,rx in [(100,90,45,40),(155,175,35,28),(200,90,30,25)]:
    y,x=np.ogrid[:H,:W]
    fire_mask[((y-cy)/ry)**2+((x-cx)/rx)**2<1.0]=True
post_nir[fire_mask]-=rng.uniform(0.25,0.42,fire_mask.sum())
post_red[fire_mask]+=rng.uniform(0.10,0.22,fire_mask.sum())
post_swir[fire_mask]+=rng.uniform(0.15,0.32,fire_mask.sum())

def bai(r,n,e=1e-6): return 1/((0.1-r)**2+(0.06-n)**2+e)
def nbr(n,s,e=1e-6): return (n-s)/(n+s+e)

pre_bai=bai(pre_red,pre_nir); post_bai=bai(post_red,post_nir)
dnbr=nbr(pre_nir,pre_swir)-nbr(post_nir,post_swir)
PIXEL_HA=(10**2)/10000
unb=(dnbr<=0.10); lo=(dnbr>0.10)&(dnbr<=0.27)
med=(dnbr>0.27)&(dnbr<=0.66); hi=(dnbr>0.66)
TOTAL=( lo|med|hi).sum()*PIXEL_HA
print(f"Burn Severity (USFS dNBR thresholds):")
print(f"  Unburned  (<0.10) : {unb.mean()*100:5.1f}%")
print(f"  Low       (0.10-0.27): {lo.mean()*100:5.1f}%  {lo.sum()*PIXEL_HA:,.0f} ha")
print(f"  Medium    (0.27-0.66): {med.mean()*100:5.1f}%  {med.sum()*PIXEL_HA:,.0f} ha")
print(f"  High      (>0.66) : {hi.mean()*100:5.1f}%  {hi.sum()*PIXEL_HA:,.0f} ha")
print(f"  TOTAL BURNED       :          {TOTAL:,.0f} ha")

## Visualisation

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(13,10))
b1=axes[0,0].imshow(pre_bai,cmap='RdYlGn_r',vmin=0,vmax=15)
axes[0,0].set_title("BAI Pre-Fire",fontsize=11,fontweight='bold'); axes[0,0].axis('off')
plt.colorbar(b1,ax=axes[0,0],fraction=0.046,pad=0.04)
b2=axes[0,1].imshow(post_bai,cmap='RdYlGn_r',vmin=0,vmax=15)
axes[0,1].set_title("BAI Post-Fire",fontsize=11,fontweight='bold'); axes[0,1].axis('off')
plt.colorbar(b2,ax=axes[0,1],fraction=0.046,pad=0.04)
dc=mcolors.ListedColormap(['#ECF0F1','#F9E79F','#E67E22','#C0392B'])
dn=mcolors.BoundaryNorm([dnbr.min(),0.10,0.27,0.66,dnbr.max()],4)
axes[1,0].imshow(dnbr,cmap=dc,norm=dn)
axes[1,0].set_title("dNBR Severity Map
USFS 4-class",fontsize=11,fontweight='bold'); axes[1,0].axis('off')
legend=[mpatches.Patch(color='#ECF0F1',label=f"Unburned {unb.mean()*100:.0f}%"),
         mpatches.Patch(color='#F9E79F',label=f"Low {lo.mean()*100:.0f}%"),
         mpatches.Patch(color='#E67E22',label=f"Medium {med.mean()*100:.0f}%"),
         mpatches.Patch(color='#C0392B',label=f"High {hi.mean()*100:.0f}%")]
axes[1,0].legend(handles=legend,loc='lower right',fontsize=9)
axes[1,1].bar(['Unburned','Low','Medium','High'],
               [unb.mean()*100,lo.mean()*100,med.mean()*100,hi.mean()*100],
               color=['#ECF0F1','#F9E79F','#E67E22','#C0392B'],edgecolor='black',lw=0.8)
axes[1,1].set_ylabel("Area (%)"); axes[1,1].set_title("Severity Distribution",fontsize=11,fontweight='bold')
plt.suptitle(f"Wildfire — Sierra Nevada, CA | {TOTAL:,.0f} ha burned",fontsize=12,fontweight='bold')
plt.tight_layout(); plt.savefig(BASE_DIR/"wildfire.png",dpi=150,bbox_inches='tight'); plt.show()

## Summary

In [ ]:
print("="*60); print("NOTEBOOK 16 — Wildfire Detection"); print("="*60)
print(f"Study : Sierra Nevada, California")
print(f"Indices: BAI + dNBR  |  USFS 4-class severity")
print(f"Burned: {TOTAL:,.0f} ha  |  High: {hi.mean()*100:.1f}%")
print("\nGeoAI: client.geoai.fire.burned_area(pre, post)")
print("Next: 17_species_biodiversity_mapping.ipynb")